In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
data = pd.read_pickle('RFM анализ таблица.pkl')
data

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score_sum,RFM_Score_str,Segment
CustomerID,,,,,,,,,
12346.0,326,1,77183.60,1,1,5,7,115,Other
12347.0,2,7,4310.00,5,5,5,15,555,Champion
12348.0,75,4,1797.24,2,4,4,10,244,At Risk
12349.0,19,1,1757.55,4,1,4,9,414,New
12350.0,310,1,334.40,1,1,2,4,112,Lost
...,...,...,...,...,...,...,...,...,...
18280.0,278,1,180.60,1,2,1,4,121,Lost
18281.0,181,1,80.82,1,2,1,4,121,Lost
18282.0,8,2,178.05,5,3,1,9,531,Other


In [3]:
def churn(churn_class):
    if churn_class > 180 :
        return 1
    else:
        return 0
data['churn'] = data['Recency'].apply(churn)
data

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score_sum,RFM_Score_str,Segment,churn
CustomerID,,,,,,,,,,
12346.0,326,1,77183.60,1,1,5,7,115,Other,1
12347.0,2,7,4310.00,5,5,5,15,555,Champion,0
12348.0,75,4,1797.24,2,4,4,10,244,At Risk,0
12349.0,19,1,1757.55,4,1,4,9,414,New,0
12350.0,310,1,334.40,1,1,2,4,112,Lost,1
...,...,...,...,...,...,...,...,...,...,...
18280.0,278,1,180.60,1,2,1,4,121,Lost,1
18281.0,181,1,80.82,1,2,1,4,121,Lost,1
18282.0,8,2,178.05,5,3,1,9,531,Other,0


In [4]:
data.value_counts('churn')

churn
0    3478
1     860
Name: count, dtype: int64

In [5]:
X = data[['Frequency','Monetary']]
Y = data['churn']
X.head()


,Frequency,Monetary
CustomerID,,
12346.0,1,77183.60
12347.0,7,4310.00
12348.0,4,1797.24
12349.0,1,1757.55
12350.0,1,334.40


In [6]:
X_train , X_test,Y_train, Y_test = train_test_split(
    X,Y,test_size=0.3,random_state=51,stratify=Y)


In [7]:
X_train.shape

(3036, 2)

In [8]:
X_test.shape


(1302, 2)

In [9]:
Y_train.value_counts(normalize=True)

churn
0    0.801713
1    0.198287
Name: proportion, dtype: float64

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced',random_state=51)
model.fit(X_train_scaled,Y_train)

y_fut = model.predict(X_test_scaled)
print(classification_report(Y_test,y_fut))

              precision    recall  f1-score   support

           0       0.97      0.55      0.70      1044
           1       0.34      0.92      0.49       258

    accuracy                           0.63      1302
   macro avg       0.65      0.74      0.60      1302
weighted avg       0.84      0.63      0.66      1302



In [11]:
rand_for = RandomForestClassifier(class_weight='balanced',random_state=51,n_estimators=100)
rand_for.fit(X_train,Y_train)
y_fut_rf = rand_for.predict(X_test)
print(classification_report(Y_test,y_fut_rf))

              precision    recall  f1-score   support

           0       0.84      0.85      0.85      1044
           1       0.37      0.37      0.37       258

    accuracy                           0.75      1302
   macro avg       0.61      0.61      0.61      1302
weighted avg       0.75      0.75      0.75      1302



In [12]:
importance = pd.Series(rand_for.feature_importances_, index=X.columns).sort_values(ascending=False)
importance

Monetary     0.826569
Frequency    0.173431
dtype: float64